# Qwen3-0.6B coherence-preserving mixed SFT

This notebook is designed for **VS Code connected to a Google Colab A100 runtime**. It combines a seeded random sample of 500 preferred responses from the old control bundle with every accepted example from the verified SVAMP/Qwen3-4B dataset, then fully fine-tunes `Qwen/Qwen3-0.6B` for one epoch.

- exactly 500 old control-training examples, sampled without replacement
- all available verified SVAMP training examples to reinforce coherent, correct reasoning
- a held-out mixture of 100 old control examples and all verified SVAMP validation examples
- rejected DPO responses are never used for SFT
- 4,096-token cutoff
- quarter-epoch evaluation/checkpoints, final weights, logs, configuration, and loss plots are saved to Google Drive

Before running, select an A100 runtime, copy `colab_qwen_dpo_bundle.zip` into the root of Google Drive (`MyDrive`), and finish the SVAMP generation notebook so its verified train/eval files exist in Drive.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"({props.total_memory / 2**30:.1f} GiB)")
assert "A100" in props.name, "Select a Colab A100 runtime for this notebook."
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

## 1. Mount Drive and unpack the existing 5K bundle

VS Code cannot use Colab's browser upload widget reliably, so the prepared zip is read from Drive.

In [ ]:
import shutil
import zipfile

from google.colab import drive

drive.mount("/content/drive")
BUNDLE = Path("/content/drive/MyDrive/colab_qwen_dpo_bundle.zip")
assert BUNDLE.is_file(), f"Copy the prepared bundle to {BUNDLE}"

PROJECT = Path("/content/qwen3_06b_5k_sft")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)

with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(PROJECT)

(PROJECT / "data").mkdir(exist_ok=True)
(PROJECT / "scripts").mkdir(exist_ok=True)
for path in PROJECT.glob("*.json"):
    shutil.move(str(path), PROJECT / "data" / path.name)
for path in PROJECT.glob("*.py"):
    shutil.move(str(path), PROJECT / "scripts" / path.name)

SOURCE = PROJECT / "data/multilingual_thinking_qwen3_4b_cots.json"
PREPARE_DPO = PROJECT / "scripts/prepare_all_dpo_data.py"
assert SOURCE.is_file() and PREPARE_DPO.is_file()
print("Workspace:", PROJECT)
print("Source size:", f"{SOURCE.stat().st_size / 2**20:.1f} MiB")

## 2. Install LLaMA-Factory

This uses the same pinned LLaMA-Factory stack as the earlier DPO experiment and preserves Colab's CUDA-enabled PyTorch.

In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "llamafactory[torch,metrics]==0.9.5",
        "transformers>=4.51,<4.57",
        "datasets>=3.2",
        "accelerate>=1.2",
        "sentencepiece",
        "protobuf",
        "tensorboard",
        "matplotlib",
        "pandas",
        "pyyaml",
    ],
    check=True,
)

import llamafactory
import transformers

print("LLaMA-Factory:", llamafactory.__version__)
print("Transformers:", transformers.__version__)

## 3. Rebuild and verify the exact 5K dataset

The existing preparation script reconstructs the three English formatting controls and registers the seven multilingual controls. The assertions prevent accidentally training on an incomplete bundle.

In [ ]:
import json

subprocess.run(
    [
        sys.executable,
        str(PREPARE_DPO),
        "--input", str(SOURCE),
        "--output-dir", str(PROJECT / "data"),
        "--eval-size", "100",
        "--seed", "42",
    ],
    check=True,
)

REGISTRY_PATH = PROJECT / "data/dataset_info.json"
registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
dpo_train_names = sorted(name for name in registry if name.endswith("_dpo_train"))
dpo_eval_names = sorted(name for name in registry if name.endswith("_dpo_eval"))
assert len(dpo_train_names) == len(dpo_eval_names) == 10

def load_registered(name):
    return json.loads(
        (PROJECT / "data" / registry[name]["file_name"]).read_text(encoding="utf-8")
    )

train_count = sum(len(load_registered(name)) for name in dpo_train_names)
eval_count = sum(len(load_registered(name)) for name in dpo_eval_names)
assert (train_count, eval_count) == (4500, 500), (train_count, eval_count)
print("Training datasets:", dpo_train_names)
print("Validation datasets:", dpo_eval_names)
print(f"Verified bundle: {train_count} train + {eval_count} validation = {train_count + eval_count}")

## 4. Build the coherence-preserving mixed SFT dataset

The old control contribution is limited to a reproducible random 500-row sample. Every available row from the recommended verified SVAMP split is added without duplication or style expansion. Rejected DPO responses are excluded.

In [ ]:
import random

REQUIRED_DPO_KEYS = {"instruction", "input", "chosen", "rejected"}
CONTROL_TRAIN_SAMPLE = 500
CONTROL_EVAL_SAMPLE = 100
MIX_SEED = 42

def control_pool(names):
    converted = []
    for name in names:
        for index, row in enumerate(load_registered(name)):
            if set(row) != REQUIRED_DPO_KEYS:
                raise ValueError(f"Unexpected schema in {name}[{index}]: {sorted(row)}")
            if not all(isinstance(row[key], str) and row[key].strip() for key in REQUIRED_DPO_KEYS):
                raise ValueError(f"Empty or non-string field in {name}[{index}]")
            converted.append({
                "instruction": row["instruction"].strip(),
                "input": row["input"].strip(),
                "output": row["chosen"].strip(),
            })
    return converted

old_control_train = control_pool(dpo_train_names)
old_control_eval = control_pool(dpo_eval_names)
rng = random.Random(MIX_SEED)
control_train = rng.sample(old_control_train, CONTROL_TRAIN_SAMPLE)
control_eval = random.Random(MIX_SEED + 1).sample(old_control_eval, CONTROL_EVAL_SAMPLE)

SVAMP_DATA_DRIVE = Path(
    "/content/drive/MyDrive/CoT_Controllability/svamp_qwen3_4b_verified_sft"
)
SVAMP_TRAIN = SVAMP_DATA_DRIVE / "svamp_verified_sft_train.json"
SVAMP_EVAL = SVAMP_DATA_DRIVE / "svamp_verified_sft_eval.json"
assert SVAMP_TRAIN.is_file() and SVAMP_EVAL.is_file(), (
    "Verified SVAMP files are missing. Finish the SVAMP trace-generation "
    f"notebook first; expected {SVAMP_TRAIN} and {SVAMP_EVAL}."
)

def load_svamp(path):
    rows = json.loads(path.read_text(encoding="utf-8"))
    converted = []
    seen_ids = set()
    for index, row in enumerate(rows):
        required = {"id", "instruction", "input", "output"}
        if not required.issubset(row):
            raise ValueError(f"Missing fields in {path.name}[{index}]")
        if row["id"] in seen_ids:
            raise ValueError(f"Duplicate SVAMP id in {path.name}: {row['id']}")
        seen_ids.add(row["id"])
        converted.append({
            "instruction": row["instruction"].strip(),
            "input": row["input"].strip(),
            "output": row["output"].strip(),
        })
    if not converted:
        raise ValueError(f"{path} is empty")
    return converted, seen_ids

svamp_train, svamp_train_ids = load_svamp(SVAMP_TRAIN)
svamp_eval, svamp_eval_ids = load_svamp(SVAMP_EVAL)
assert not (svamp_train_ids & svamp_eval_ids), "SVAMP IDs overlap across train and eval"

sft_train = control_train + svamp_train
sft_eval = control_eval + svamp_eval
random.Random(MIX_SEED + 2).shuffle(sft_train)
random.Random(MIX_SEED + 3).shuffle(sft_eval)

# No identical instruction/problem pair may cross the held-out boundary.
train_prompts = {(row["instruction"], row["input"]) for row in sft_train}
eval_prompts = {(row["instruction"], row["input"]) for row in sft_eval}
overlap = train_prompts & eval_prompts
assert not overlap, f"Found {len(overlap)} train/eval prompt overlaps"

SFT_TRAIN_FILE = PROJECT / "data/control500_plus_svamp_sft_train.json"
SFT_EVAL_FILE = PROJECT / "data/control500_plus_svamp_sft_eval.json"
SFT_TRAIN_FILE.write_text(json.dumps(sft_train, ensure_ascii=False, indent=2), encoding="utf-8")
SFT_EVAL_FILE.write_text(json.dumps(sft_eval, ensure_ascii=False, indent=2), encoding="utf-8")

sft_registry = {
    "control500_plus_svamp_sft_train": {
        "file_name": SFT_TRAIN_FILE.name,
        "columns": {"prompt": "instruction", "query": "input", "response": "output"},
    },
    "control500_plus_svamp_sft_eval": {
        "file_name": SFT_EVAL_FILE.name,
        "columns": {"prompt": "instruction", "query": "input", "response": "output"},
    },
}
registry.update(sft_registry)
REGISTRY_PATH.write_text(json.dumps(registry, indent=2) + "\n", encoding="utf-8")

SFT_DATA_DRIVE = Path(
    "/content/drive/MyDrive/CoT_Controllability/control500_plus_svamp_sft_data"
)
SFT_DATA_DRIVE.mkdir(parents=True, exist_ok=True)
shutil.copy2(SFT_TRAIN_FILE, SFT_DATA_DRIVE / SFT_TRAIN_FILE.name)
shutil.copy2(SFT_EVAL_FILE, SFT_DATA_DRIVE / SFT_EVAL_FILE.name)
(SFT_DATA_DRIVE / "dataset_info.json").write_text(
    json.dumps(sft_registry, indent=2) + "\n", encoding="utf-8"
)
manifest = {
    "seed": MIX_SEED,
    "control_train": len(control_train),
    "svamp_train": len(svamp_train),
    "total_train": len(sft_train),
    "control_eval": len(control_eval),
    "svamp_eval": len(svamp_eval),
    "total_eval": len(sft_eval),
    "rejected_responses_used": 0,
}
(SFT_DATA_DRIVE / "mixture_manifest.json").write_text(
    json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
)

print(json.dumps(manifest, indent=2))
print("Persistent mixed SFT dataset:", SFT_DATA_DRIVE)
print("Example instruction:", sft_train[0]["instruction"])
print("Example target prefix:", sft_train[0]["output"][:160].replace("\n", " "))

## 5. Write the full-SFT configuration

A micro-batch of two with four accumulation steps gives an effective batch size of eight. The learning rate is reduced to `5e-6`, matching the conservative ReasonIF RIF setting. Evaluation and checkpoints occur at roughly 0.25, 0.50, 0.75, and 1.00 epoch so the most coherent checkpoint can be selected.

In [ ]:
import math
import yaml

MODEL_NAME = "Qwen/Qwen3-0.6B"
CUTOFF_LEN = 4096
MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
EPOCHS = 1.0
EXPECTED_STEPS = math.ceil(len(sft_train) / (MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION))
QUARTER_EPOCH_STEPS = max(1, math.ceil(EXPECTED_STEPS / 4))

DRIVE_OUTPUT = Path(
    "/content/drive/MyDrive/CoT_Controllability/"
    "qwen3-0.6b-control500-plus-svamp-sft-cutoff4096-1epoch"
)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

config = {
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "full",
    "model_name_or_path": MODEL_NAME,
    "dataset_dir": str(PROJECT / "data"),
    "dataset": "control500_plus_svamp_sft_train",
    "eval_dataset": "control500_plus_svamp_sft_eval",
    "template": "qwen3",
    "cutoff_len": CUTOFF_LEN,
    "train_on_prompt": False,
    "overwrite_cache": True,
    "preprocessing_num_workers": 4,
    "output_dir": str(DRIVE_OUTPUT),
    "overwrite_output_dir": False,
    "report_to": "tensorboard",
    "logging_steps": 5,
    "disable_tqdm": False,
    "plot_loss": True,
    "per_device_train_batch_size": MICRO_BATCH_SIZE,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
    "gradient_checkpointing": True,
    "bf16": True,
    "tf32": True,
    "learning_rate": 5.0e-6,
    "num_train_epochs": EPOCHS,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "max_grad_norm": 1.0,
    "eval_strategy": "steps",
    "eval_steps": QUARTER_EPOCH_STEPS,
    "eval_on_start": True,
    "save_strategy": "steps",
    "save_steps": QUARTER_EPOCH_STEPS,
    "save_total_limit": 5,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "seed": 42,
    "data_seed": 42,
}

checkpoints = sorted(
    DRIVE_OUTPUT.glob("checkpoint-*"),
    key=lambda path: int(path.name.split("-")[-1]),
)
if checkpoints:
    config["resume_from_checkpoint"] = str(checkpoints[-1])
    print("Will resume from:", checkpoints[-1])

CONFIG_PATH = PROJECT / "qwen3_06b_control500_plus_svamp_sft.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
(DRIVE_OUTPUT / "training_config.yaml").write_text(
    CONFIG_PATH.read_text(encoding="utf-8"), encoding="utf-8"
)
print(f"Expected optimizer steps: approximately {EXPECTED_STEPS}")
print(f"Evaluate and save every {QUARTER_EPOCH_STEPS} steps")
print("Output directory:", DRIVE_OUTPUT)
print(CONFIG_PATH.read_text(encoding="utf-8"))

## 6. Train with visible progress

The LLaMA-Factory progress bar appears directly in this cell. Validation runs before training and at each quarter-epoch checkpoint. Checkpoints and the final model are written to Drive. If the runtime stops after a checkpoint is complete, reconnect and rerun from the top; the configuration cell selects the newest checkpoint automatically.

In [ ]:
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"

command = [sys.executable, "-m", "llamafactory.cli", "train", str(CONFIG_PATH)]
print("Running:", " ".join(command), flush=True)
result = subprocess.run(command, cwd=PROJECT, env=env)
if result.returncode != 0:
    raise RuntimeError(f"LLaMA-Factory exited with status {result.returncode}")
print("Training completed successfully.", flush=True)

## 7. Inspect saved weights and compare training/validation loss

SFT does not have DPO pairwise accuracy. Its direct learning signal is token-level cross-entropy loss, so this section reports training loss and held-out validation loss without inventing an accuracy metric.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

state_candidates = list(DRIVE_OUTPUT.glob("checkpoint-*/trainer_state.json"))
if (DRIVE_OUTPUT / "trainer_state.json").is_file():
    state_candidates.append(DRIVE_OUTPUT / "trainer_state.json")
assert state_candidates, "No trainer_state.json was found. Training may not have completed."
STATE_PATH = max(state_candidates, key=lambda path: path.stat().st_mtime)
state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
history = pd.DataFrame(state["log_history"])

train_history = history[history["loss"].notna()][["step", "epoch", "loss"]].copy()
eval_history = history[history["eval_loss"].notna()][["step", "epoch", "eval_loss"]].copy()
display(eval_history.reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(train_history["step"], train_history["loss"], alpha=0.35, label="logged train loss")
if len(train_history) >= 5:
    axes[0].plot(
        train_history["step"], train_history["loss"].rolling(5, min_periods=1).mean(),
        linewidth=2, label="5-log moving average",
    )
axes[0].set(title="SFT training loss", xlabel="optimizer step", ylabel="cross-entropy loss")
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(eval_history["epoch"], eval_history["eval_loss"], marker="o")
axes[1].set(title="Held-out validation loss", xlabel="epoch", ylabel="cross-entropy loss")
axes[1].grid(alpha=0.2)
fig.tight_layout()
LOSS_PLOT = DRIVE_OUTPUT / "sft_train_validation_loss.png"
fig.savefig(LOSS_PLOT, dpi=160, bbox_inches="tight")
plt.show()

weight_files = list(DRIVE_OUTPUT.glob("*.safetensors")) + list(DRIVE_OUTPUT.glob("pytorch_model*.bin"))
checkpoint_dirs = sorted(DRIVE_OUTPUT.glob("checkpoint-*"))
print("Trainer state:", STATE_PATH)
print("Final weight files:", [path.name for path in weight_files])
print("Checkpoint directories:", [path.name for path in checkpoint_dirs])
print("Loss plot:", LOSS_PLOT)

## 8. TensorBoard (optional)

Run this during or after training for the complete metric history.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/CoT_Controllability/qwen3-0.6b-control500-plus-svamp-sft-cutoff4096-1epoch/runs

## 9. Release the Colab GPU when finished (optional)

Only run this after inspecting the results. It disconnects and releases the assigned runtime.

In [ ]:
from google.colab import runtime
runtime.unassign()